In [10]:
import sleap
import cv2
import numpy as np
import sleap_io as sio

# Load from SLEAP file.
predictions = sio.load_file("labels.v001-cs-000.slp")


# Extract video dimensions and frame count via shape tuple (frames, height, width, channels)
if predictions.videos:
    video_obj = predictions.videos[0]
    total_frames, height, width = video_obj.shape[0], video_obj.shape[1], video_obj.shape[2]
#else:
#    # Fallback to coordinate max if video metadata is unlinked
#    all_pts = [inst.numpy() for lf in predictions for inst in lf.instances]
#    all_pts = np.vstack(all_pts) if all_pts else np.zeros((1, 2))
#    width = int(np.nanmax(all_pts[:, 0])) + 50 if len(all_pts) else 1024
#    height = int(np.nanmax(all_pts[:, 1])) + 50 if len(all_pts) else 1024
#    total_frames = max(lf.frame_idx for lf in predictions) + 1

# Map labeled frames by their true video frame index
frame_map = {lf.frame_idx: lf for lf in predictions}

# Pre-extract skeleton topology
skeleton = predictions.skeletons[0]
edges = [(skeleton.nodes.index(e.source), skeleton.nodes.index(e.destination)) for e in skeleton.edges]

fps = 10
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('skeleton_only_video.mp4', fourcc, fps, (width, height))

# 2. Render frames
for frame_idx in range(total_frames):
    # White background (255 for all BGR channels)
    img = np.full((height, width, 3), 255, dtype=np.uint8)
    
    if frame_idx in frame_map:
        frame_data = frame_map[frame_idx]
        
        for instance in frame_data.instances:
            pts = instance.numpy()

            # Draw skeleton lines (Black: BGR = 0, 0, 0)
            for idx1, idx2 in edges:
                p1, p2 = pts[idx1], pts[idx2]
                if not (np.isnan(p1).any() or np.isnan(p2).any()):
                    pt1 = (int(round(p1[0])), int(round(p1[1])))
                    pt2 = (int(round(p2[0])), int(round(p2[1])))
                    
                    if 0 <= pt1[0] < width and 0 <= pt1[1] < height and \
                       0 <= pt2[0] < width and 0 <= pt2[1] < height:
                        cv2.line(img, pt1, pt2, (0, 0, 0), 2, cv2.LINE_AA)

            # Draw nodes (Red points: BGR = 0, 0, 255)
            for p in pts:
                if not np.isnan(p).any():
                    center = (int(round(p[0])), int(round(p[1])))
                    if 0 <= center[0] < width and 0 <= center[1] < height:
                        cv2.circle(img, center, 4, (0, 0, 255), -1, cv2.LINE_AA)
    out.write(img)

out.release()
#print(f"Video rendered successfully! Dimensions: {width}x{height}, Total Frames: {total_frames}")
